### Mini‑Project: Customer Support Assistant

**Goal**

Build a chain that **classifies** a user message into an intent, **routes** it to a specialist, and returns a **structured response**.

**Components Used**

- **Few‑shot prompting** – for intent classification.
- **`PydanticOutputParser`** – for structured output (`SupportResponse`).
- **`RunnableBranch`** – for routing to refund/technical/general chains.
- **`RunnableLambda`** – for custom classification and response creation.
- **`.with_config()`** – for metadata and tags.
- **`.with_retry()`** – for reliability.

**Flow**

1. User message → classify intent (refund, technical, general).
2. Route to the corresponding specialist prompt/chain.
3. Generate an answer using the specialist chain.
4. Combine intent, answer, and confidence into a `SupportResponse` object.
5. Display the result beautifully.

**In This Notebook**

- We build the classification chain.
- We define the specialist prompts and chains.
- We combine everything into a production‑ready chain.
- We test with multiple messages and display the structured result.

Step 1: Imports and Environment

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import json

# Load environment variables from .env
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

Step 2: Define the Output Model

In [12]:
# Define the structured output we want from the support assistant
class SupportResponse(BaseModel):
    intent: str = Field(description='The type of request: refund, technical, or general')
    answer: str = Field(description="The assistant's reply to the user")
    confidence: float = Field(description='Confidence score between 0 and 1 (placeholder)')
    

Step 3: Create the Parser and Format Instructions

In [13]:
# Create the parser from the Pydantic model
parser = PydanticOutputParser(pydantic_object=SupportResponse)

# Extract JSON formatting directives
format_instructions = parser.get_format_instructions()

print('Format instructions (first 200 chars):')
print(f"{format_instructions[:200]}, '...'")

Format instructions (first 200 chars):
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "ty, '...'


Step 4: Build Intent Classification Chain

In [14]:
# Create the chat model
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Define few-shot examples for intent classification
intent_examples = [
    {'message': 'I want a efund for a broken laptop.', 'intent': 'refund'},
    {'message': 'My laptop keeps crashing.', 'intent': 'technical'},
    {'message': 'What is your return policy', 'intent': 'general'}
]

# Format each example as human (message) / AI (intent)
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{message}'),
    ('ai', '{intent}')
])

# Combine examples into a few-shot prompt; expects 'message' from input
few_shot_intent_prompt = FewShotChatMessagePromptTemplate(
    examples=intent_examples,
    example_prompt=example_prompt,
    input_variables=['message']
)

# Final prompt: system instruction + examples + new message
intent_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Classify the user message into one of these intents: refund, technical, or general. Return only the intent word.'),
    few_shot_intent_prompt,
    ('human', '{message}')
])

# Chain: prompt -> model -> parser (returns intent string)
intent_chain = intent_prompt | llm | StrOutputParser()

# Test the classification chain with a new message
test_intent = intent_chain.invoke({
    'message': 'I need help with my screen flickering.'
})

print(f'Classification intent: {test_intent}')

Classification intent: technical


Step 5: Define Specialist Chains(for each intent: refund, technical and general)


In [15]:
# Refund specialist prompt
refund_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a refund specialist. Help the user with their refund request.'),
    ('human', '{message}')
])

# Technical support specialist prompt
technical_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a technical support specialist. Provide troubleshooting steps.'),
    ('human', '{message}')
])

# General assistant prompt
general_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a general customer support assistant. Answer the user politely.'),
    ('human', '{message}')
])

# Build each chain: prompt -> model -> StrOutputParser (returns string answer)
refund_chain = refund_prompt | llm | StrOutputParser()
technical_chain = technical_prompt | llm | StrOutputParser()
general_chain = general_prompt | llm | StrOutputParser()

Step 6: Create Classification Step and Branch

In [16]:
# Wrap classification into a reusable step
def classify_message(message):
    # Classify intent using the intent_chain
    intent = intent_chain.invoke({'message': message})
    
    # Return both message and intent for the branch
    return {'message': message, 'intent': intent}
    
classify_step = RunnableLambda(classify_message)

# Build route to the right specialist chain based on intent
specialist_branch = RunnableBranch(
    (lambda x: x['intent'] == 'refund', refund_chain),
    (lambda x: x['intent'] == 'technical', technical_chain),
    general_chain   # default fallback
)

Step 7: Build Final Structured Chain

In [18]:
def create_support_response(inputs):
    # Extract message and classified intent
    message = inputs['message']
    intent = inputs['intent']
    
    # Get answer from the branch (specialist chain)
    answer = specialist_branch.invoke(inputs)
    
    # Return a SupportResponse object
    return SupportResponse(intent=intent, answer=answer, confidence=0.9)

# Build final chain: classify -> branch -> structured response
final_chain = classify_step | RunnableLambda(create_support_response)

# Test the final chain
result = final_chain.invoke({'message': 'I need help with my screen flickering.'})

print('Final structured result:')
print(result)
print('Type:', type(result))

Final structured result:
intent='technical' answer='Screen flickering can be caused by various issues, including hardware problems, software conflicts, or driver issues. Here are some troubleshooting steps you can follow to identify and resolve the problem:\n\n### Step 1: Check the Monitor Connection\n1. **Inspect Cables**: Ensure that the monitor cables (HDMI, DisplayPort, VGA, etc.) are securely connected to both the monitor and the computer.\n2. **Try a Different Cable**: If possible, replace the cable with a different one to rule out a faulty cable.\n3. **Test with Another Monitor**: If you have access to another monitor, connect it to your computer to see if the flickering persists.\n\n### Step 2: Adjust Display Settings\n1. **Refresh Rate**: Right-click on the desktop and select "Display settings." Scroll down and click on "Advanced display settings." Check the refresh rate and ensure it is set to the recommended value for your monitor.\n2. **Resolution**: Ensure that the display

Step 8: Add Config and Retry to Final Chain

In [24]:
# Make the final chain more robust with retry and config
production_chain = final_chain.with_config(
    run_name='CustomerSupportAssistant',
    metadata={'module': '2B', 'type': 'mini_project'},
    tags=['customer_support', 'few_short', 'branch']
).with_retry(
    stop_after_attempt=2,
    wait_exponential_jitter=True
)

print('Production-ready chain created successfully.')

Production-ready chain created successfully.


In [28]:
import pandas as pd
from IPython.display import display

def show_support_response(res: SupportResponse):
    
    df_metadata = pd.DataFrame({
        'Field': ['Intent', 'Confidence'],
        'Value': [res.intent, f'{res.confidence:.2f}']
    })
    
    # Display the metadata table
    print('Support Response')
    display(df_metadata)
    
    print("\nAnswer:")
    print(res.answer)

In [30]:
# # Test with a Single Message
# response = production_chain.invoke({
#     'message': 'I need help with my screen flickering.'
# })

# show_support_response(response)

In [33]:
import re

def clean_markdown_text(text):
    # Remove bold/italic markers
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    # Remove headings (###) but keep the heading text
    text = re.sub(r'^###\s+', '', text, flags=re.MULTILINE)
    # Remove other common markdown symbols (e.g., backticks, hashes)
    text = re.sub(r'`', '', text)
    # Replace multiple blank lines with one
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def show_support_response_clean(res: SupportResponse):
    print("Support Response:")
    print(f"Intent: {res.intent}")
    print(f"Confidence: {res.confidence:.2f}")
    print("\nAnswer:")
    cleaned = clean_markdown_text(res.answer)
    print(cleaned)

# Test
response = production_chain.invoke({'message': 'I need help with my screen flickering.'})
show_support_response_clean(response)

Support Response:
Intent: technical
Confidence: 0.90

Answer:
Screen flickering can be caused by various issues, including hardware problems, software conflicts, or display settings. Here are some troubleshooting steps you can follow to identify and resolve the issue:

Step 1: Check the Connections
1. Inspect Cables: Ensure that all cables connecting your monitor to your computer are secure and undamaged. This includes power cables and video cables (HDMI, DisplayPort, VGA, etc.).
2. Try a Different Port: If your monitor has multiple input ports, try connecting it to a different port on your computer.

Step 2: Update Graphics Drivers
1. Open Device Manager: Right-click on the Start menu and select "Device Manager."
2. Expand Display Adapters: Find your graphics card in the list and right-click on it.
3. Update Driver: Select "Update driver" and follow the prompts to search for updated drivers automatically.
4. Restart Your Computer: After updating, restart your computer to see if the fl